# preprocessing.ipynb

This notebook prepares raw `.npz` data for YOLOv8 training. It contains:
- `mask_to_yolo_boxes_multi`: converts binary segmentation masks (H, W) into YOLO-format bounding boxes (class\_id, x\_center, y\_center, width, height) normalized to image dimensions, using connected-component labeling.
- A script that reads all `.npz` files (each containing a 3-channel image and 7-class masks), extracts bounding boxes for each class, and saves them to a CSV file.
- A script that reads the CSV, splits data into train/val (80/20), converts `.npz` images to PNG files, writes YOLO-format `.txt` label files, and generates a `data.yaml` configuration file.

In [ ]:
import numpy as np
from scipy.ndimage import label

def mask_to_yolo_boxes_multi(single_mask, class_id, img_w, img_h, connectivity=1):
    """
    single_mask: (H, W) binary/boolean mask (may contain multiple objects).
    Returns list of [class_id, x_c_n, y_c_n, w_n, h_n].
    """
    labeled, num = label(single_mask, structure=None if connectivity == 1 else np.ones((3, 3)))
    boxes = []

    for comp_id in range(1, num + 1):
        ys, xs = np.where(labeled == comp_id)
        if len(xs) == 0:
            continue

        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()

        box_w  = x_max - x_min + 1
        box_h  = y_max - y_min + 1
        x_c    = x_min + box_w / 2.0
        y_c    = y_min + box_h / 2.0

        x_c_n = x_c / img_w
        y_c_n = y_c / img_h
        w_n   = box_w / img_w
        h_n   = box_h / img_h

        boxes.append([class_id, x_c_n, y_c_n, w_n, h_n])

    return boxes

In [ ]:
import os
import glob
import csv
import numpy as np

from scipy.ndimage import label  # ensure imported
# mask_to_yolo_boxes_multi assumed defined above

DATA_DIR = "../../data/MR/data/processed/tiles/marius_hills/"       # folder containing *.npz
OUT_CSV  = "all_boxes.csv"

npz_paths = glob.glob(os.path.join(DATA_DIR, "*.npz"))

all_rows = []  # each row: [npz_path, class_id, x_center, y_center, width, height]

for npz_path in npz_paths:
    data = np.load(npz_path)
    img   = data["image"]   # (3, H, W)
    masks = data["mask"]    # (7, H, W)

    _, H, W = img.shape

    for class_id in range(masks.shape[0]):  # 0..6
        m = masks[class_id] > 0  # boolean
        boxes = mask_to_yolo_boxes_multi(m, class_id, W, H)

        for (cls, x_c, y_c, w, h) in boxes:
            all_rows.append([npz_path, cls, x_c, y_c, w, h])

# Write single global CSV
with open(OUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["npz_path", "class_id", "x_center", "y_center", "width", "height"])
    writer.writerows(all_rows)

print(f"Saved {len(all_rows)} boxes from {len(npz_paths)} npz files to {OUT_CSV}")

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

CSV_PATH = "all_boxes.csv"
OUT_ROOT = "yolo_dataset"

CLASS_NAMES = {
    0: "class0",
    1: "class1",
    2: "class2",
    3: "class3",
    4: "class4",
    5: "class5",
    6: "class6",
}

df = pd.read_csv(CSV_PATH)
image_ids = df["npz_path"].unique().tolist()

train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

for split in ["train", "val"]:
    os.makedirs(os.path.join(OUT_ROOT, "images", split), exist_ok=True)
    os.makedirs(os.path.join(OUT_ROOT, "labels", split), exist_ok=True)

def save_one_image_and_label(npz_path, split):
    rows = df[df["npz_path"] == npz_path]

    data = np.load(npz_path)
    img = data["image"]  # (3, H, W)

    img_hwc = np.transpose(img, (1, 2, 0))
    if img_hwc.dtype != np.uint8:
        if img_hwc.max() <= 1.0:
            img_hwc = (img_hwc * 255).clip(0, 255).astype(np.uint8)
        else:
            img_hwc = img_hwc.clip(0, 255).astype(np.uint8)

    base = os.path.splitext(os.path.basename(npz_path))[0]
    image_path = os.path.join(OUT_ROOT, "images", split, base + ".png")
    label_path = os.path.join(OUT_ROOT, "labels", split, base + ".txt")

    Image.fromarray(img_hwc).save(image_path)

    with open(label_path, "w") as f:
        for _, row in rows.iterrows():
            f.write(
                f"{int(row['class_id'])} "
                f"{float(row['x_center']):.6f} "
                f"{float(row['y_center']):.6f} "
                f"{float(row['width']):.6f} "
                f"{float(row['height']):.6f}\n"
            )

for npz_path in train_ids:
    save_one_image_and_label(npz_path, "train")

for npz_path in val_ids:
    save_one_image_and_label(npz_path, "val")

yaml_text = f"""path: {os.path.abspath(OUT_ROOT)}
train: images/train
val: images/val
names:
""" + "\n".join([f"  {k}: {v}" for k, v in CLASS_NAMES.items()])

with open(os.path.join(OUT_ROOT, "data.yaml"), "w") as f:
    f.write(yaml_text)